# DENUE Yucatán profiling

This notebook profiles the complete INEGI DENUE state file before the spatial join. All columns are read as strings with the published Latin-1 encoding so identifiers keep their leading zeros.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.transform.denue import activity_group, sector_code

DENUE_FILE = ROOT / 'data/raw/denue_31/conjunto_de_datos/denue_inegi_31_.csv'
denue = pd.read_csv(DENUE_FILE, encoding='latin-1', dtype=str)
print(f'Rows: {len(denue):,}; columns: {len(denue.columns)}')
display(denue.head())

Rows: 146,384; columns: 42


,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,...,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,12319834,31059114119005802000000000U4,NaN,NaN,114119,"Pesca y captura de peces, crustáceos, moluscos...",6 a 10 personas,CALLE,116 16,CALLE,...,Chelem ...,0696,006,9997403666,NaN,NaN,Fijo,21.26814748,-89.74021464,2026-04
1,12319961,31038114119001743000000000U8,NaN,NaN,114119,"Pesca y captura de peces, crustáceos, moluscos...",11 a 30 personas,CALLE,NINGUNO,CALLE,...,Sisal ...,0126,115,NaN,NaN,NaN,Fijo,21.16169285,-90.0458364,2026-04
2,12319997,31059114119005932000000000U9,NaN,NaN,114119,"Pesca y captura de peces, crustáceos, moluscos...",6 a 10 personas,CALLE,17,CALLE,...,Progreso ...,0728,015,9992655032,VICTORZACARIAS@HOTMAIL.COM,NaN,Fijo,21.26881611,-89.7513355,2026-04
3,12319894,31059114119005921000000000U2,NaN,NaN,114119,"Pesca y captura de peces, crustáceos, moluscos...",0 a 5 personas,CALLE,16,CALLE,...,Progreso ...,0643,003,NaN,NaN,NaN,Fijo,21.28871964,-89.62855048,2026-04
4,12296782,31059112515000021000000000U7,NaN,NaN,112515,Piscicultura en agua dulce,0 a 5 personas,CALLE,NINGUNO,CALLE,...,Progreso ...,0713,010,NaN,NaN,NaN,Fijo,21.27725386,-89.70142094,2026-04


## Identifiers and coordinates

Duplicate `id` values would violate the fact-table grain. Coordinates are converted only for profiling here; the production transform performs the authoritative validation and reprojection.

In [2]:
coordinates = denue[['latitud', 'longitud']].apply(pd.to_numeric, errors='coerce')
coordinate_profile = pd.DataFrame({
    'null_or_invalid': coordinates.isna().sum(),
    'zero': coordinates.eq(0).sum(),
    'minimum': coordinates.min(),
    'maximum': coordinates.max(),
})
print(f"Duplicated id values: {denue['id'].duplicated().sum():,}")
display(coordinate_profile)

Duplicated id values: 0


,null_or_invalid,zero,minimum,maximum
latitud,0,0,18.687629,21.610257
longitud,0,0,-90.555357,-87.557637


## SCIAN sectors and analytical groups

SCIAN sectors 31–33 (manufacturing) and 48–49 (transportation, postal services and warehousing) are official combined sectors. Sector 46 is **Retail**. Sectors 51–56, 61, 62, 71, 72 and 81 are **Services**. All remaining sectors are **Other**; sector 93 is excluded from Services because it represents government activities. Sector names in the warehouse are stored in English.

In [3]:
denue['sector_code'] = denue['codigo_act'].map(sector_code)
denue['activity_group'] = denue['sector_code'].map(activity_group)
sector_table = (denue.groupby(['sector_code', 'activity_group'], dropna=False)
                .size().rename('establishments').reset_index())
display(sector_table)
display(denue['activity_group'].value_counts().rename_axis('activity_group').to_frame('establishments'))

,sector_code,activity_group,establishments
0,11,Other,1346
1,21,Other,42
2,22,Other,218
3,23,Other,892
4,31-33,Other,29020
5,43,Other,3681
6,46,Retail,48962
7,48-49,Other,786
8,51,Services,531
9,52,Services,1327


,establishments
activity_group,
Services,60010
Retail,48962
Other,37412


## Employment bands

`per_ocu` is an employment-size band, not an exact employee count. The transform preserves the published Spanish labels because `dw.dim_business_size` uses them as natural keys.

In [4]:
display(denue['per_ocu'].value_counts(dropna=False).rename_axis('per_ocu').to_frame('establishments'))

,establishments
per_ocu,
0 a 5 personas,126935
6 a 10 personas,10145
11 a 30 personas,6713
31 a 50 personas,1110
51 a 100 personas,754
101 a 250 personas,477
251 y más personas,250


## Registration dates

`fecha_alta` records the establishment's DENUE registration/start month; it is not the publication date of the current DENUE snapshot. The warehouse converts valid `YYYY-MM` values to the first day of that month solely to connect them to `dim_date`.

In [5]:
alta = pd.to_datetime(denue['fecha_alta'], format='%Y-%m', errors='coerce')
print(f'Valid registration months: {alta.notna().sum():,}')
print(f'Missing/invalid registration months: {alta.isna().sum():,}')
print(f'Range: {alta.min():%Y-%m} to {alta.max():%Y-%m}')
display(denue['fecha_alta'].value_counts(dropna=False).sort_index().tail(15).to_frame('establishments'))

Valid registration months: 146,375
Missing/invalid registration months: 9
Range: 2010-07 to 2026-04


,establishments
fecha_alta,
2018-11,99
2019-04,426
2019-11,26413
2020-04,894
2020-11,766
2021-05,983
2021-11,247
2022-05,14
2022-11,51


## Production decision

The ETL sends the complete state file through coordinate validation and point-in-polygon assignment. It does **not** pre-filter `cve_mun='050'`: geometry determines study-area membership, while DENUE's reported municipality, locality and AGEB codes are retained as an audit field.